# PDF Document Inspection

This notebook checks document size, page count, and text extraction quality before moving on to chunking and retrieval.

In [1]:
from pathlib import Path

import re

import fitz  # PyMuPDF

In [2]:
pdf_candidates = [
    Path("RAG Without Frameworks/knowledge-base/AWS Certified Cloud Practitioner Slides.pdf"),
    Path("knowledge-base/AWS Certified Cloud Practitioner Slides.pdf"),
]

for candidate in pdf_candidates:
    if candidate.is_file():
        pdf_path = candidate
        break
else:
    raise FileNotFoundError("Could not find the PDF in the expected knowledge-base folders.")

print("Using PDF:", pdf_path)

Using PDF: knowledge-base/AWS Certified Cloud Practitioner Slides.pdf


In [3]:
def extract_pages(path: str, start_page: int = 0) -> list[dict[str, object]]:
    """Extract page text with page metadata preserved."""
    with fitz.open(path) as pdf:
        documents = []
        for page_num, page in enumerate(pdf[start_page:], start=start_page + 1):
            documents.append({
                "page": page_num,
                "text": page.get_text(),
            })
    return documents


skip_pages = 9
# The first 9 pages are speaker-introduction pages, so we skip them before extraction.
with fitz.open(str(pdf_path)) as pdf:
    print("Pages:", len(pdf))
    print("Skipping first pages:", skip_pages)

documents = extract_pages(str(pdf_path), start_page=skip_pages)
text = "\n".join(document["text"] for document in documents)
print("Pages extracted:", len(documents))
print("First extracted page:", documents[0]["page"] if documents else None)
print("Last extracted page:", documents[-1]["page"] if documents else None)
print(f"Characters after skipping first {skip_pages} pages: {len(text)}")

Pages: 493
Skipping first pages: 9
Pages extracted: 484
First extracted page: 10
Last extracted page: 493
Characters after skipping first 9 pages: 220201


# Preserve Page Metadata

Instead of collapsing the PDF into a single string immediately, the notebook now keeps a per-page record in `documents`.

Each item stores:

- the source page number
- the extracted text for that page

This is useful later because any chunk can be traced back to something like `Source: Page 143` during retrieval or answer citation.

# Clean Repeated Boilerplate

The extracted PDF text still includes repeated footer and copyright strings on many pages. Those phrases do not help retrieval, so we remove them before sampling or chunking.

This step keeps the notebook focused on useful content and reduces noise in the final embeddings.

We also normalize whitespace and strip URLs so the cleaned text reads like plain content instead of layout artifacts.

In [4]:
import re

# These phrases repeat across the slides and act like boilerplate noise in RAG.
remove_phrases = [
    "\u00a9 Stephane Maarek",
    "NOT FOR DISTRIBUTION",
    "www.datacumulus.com",
]


def clean_text(text: str) -> str:
    """Remove repeated boilerplate, URLs, and excess whitespace."""
    for phrase in remove_phrases:
        text = text.replace(phrase, "")

    text = re.sub(r"https?://\\S+", "", text)
    text = re.sub(r"\\n+", "\\n", text)
    text = re.sub(r"\\s+", " ", text)

    lines = []
    for line in text.splitlines():
        line = line.strip()
        if line:
            lines.append(line)

    return "\n".join(lines)


# Count the repeated phrases first so the cleaning step is easy to verify.
for phrase in remove_phrases:
    print(f"{phrase}: {text.count(phrase)}")

cleaned_text = clean_text(text)
print("Original characters:", len(text))
print("Cleaned characters:", len(cleaned_text))
print("\n--- Cleaned Sample ---")
print(cleaned_text[:1000])

© Stephane Maarek: 968
NOT FOR DISTRIBUTION: 484
www.datacumulus.com: 484
Original characters: 220201
Cleaned characters: 181470

--- Cleaned Sample ---
What is Cloud Computing
Section
How websites work
Server
network
Clients have IP addresses
Servers have IP addresses
Client
Just like when you’re sending post mail!
What is a server composed of?
• Compute: CPU
• Memory: RAM
• Storage: Data
• Database: Store data in a structured way
• Network: Routers, switch, DNS server
+
=
IT Terminology
• Network: cables, routers and servers connected with each other
• Router: A networking device that forwards data packets between computer
networks. They know where to send your packets on the internet!
• Switch: Takes a packet and send it to the correct server / client on your network
Router
Switch
Traditionally, how to build infrastructure
Data center
Home or Garage
Office
Problems with traditional IT approach
• Pay for the rent for the data center
• Pay for power supply, cooling, maintenance
• Addi

# Check Cleaned Text Quality

Now inspect the cleaned text, not the raw extraction. This confirms that the repeated boilerplate was removed and that the remaining content is readable enough for chunking.

If the samples still look noisy, the cleaning rules should be expanded before moving on.

In [5]:
text_to_inspect = cleaned_text

print(text_to_inspect[:1000])

print("\n--- Sample: 10000:11000 ---")
print(text_to_inspect[10000:11000])

print("\n--- Sample: 50000:51000 ---")
print(text_to_inspect[50000:51000])

print("\n--- Page Metadata Preview ---")
print([
    {
        "page": document["page"],
        "text_chars": len(document["text"]),
        "text_preview": document["text"][:120],
    }
    for document in documents[:3]
])

What is Cloud Computing
Section
How websites work
Server
network
Clients have IP addresses
Servers have IP addresses
Client
Just like when you’re sending post mail!
What is a server composed of?
• Compute: CPU
• Memory: RAM
• Storage: Data
• Database: Store data in a structured way
• Network: Routers, switch, DNS server
+
=
IT Terminology
• Network: cables, routers and servers connected with each other
• Router: A networking device that forwards data packets between computer
networks. They know where to send your packets on the internet!
• Switch: Takes a packet and send it to the correct server / client on your network
Router
Switch
Traditionally, how to build infrastructure
Data center
Home or Garage
Office
Problems with traditional IT approach
• Pay for the rent for the data center
• Pay for power supply, cooling, maintenance
• Adding and replacing hardware takes time
• Scaling is limited
• Hire 24/7 team to monitor the infrastructure
• How to deal with disasters? (earthquake, power

# Understand the RAG Flow

PDF -> Text -> Chunks -> Embeddings -> Vectors -> Similarity Search

Chunking solves the problem of large documents by splitting them into smaller, retrievable pieces that fit within model limits and support efficient retrieval-augmented generation.

At this point in the notebook, the text has already been extracted, cleaned, and validated. The next practical step would be to split it into chunks and build retrieval-ready embeddings.

Example flow:

- PDF
- Text
- Chunks
- Embeddings
- Vectors
- Similarity Search
- Best Chunks for the question

In [10]:
# Global fixed-size chunking across the entire cleaned document (word-safe)
chunk_size = 800
chunk_overlap = 200

import re

# Build cleaned pages and offsets
cleaned_pages = [clean_text(doc['text']) for doc in documents]
offsets = []
cursor = 0
for p in cleaned_pages:
    offsets.append(cursor)
    cursor += len(p) + 1

joined = "\n".join(cleaned_pages)

chunks = []
start = 0
max_extra = 400  # allow this many extra chars to finish the current word
step = chunk_size - chunk_overlap
if step <= 0:
    step = max(1, chunk_size // 2)

while start < len(joined):
    desired_end = start + chunk_size
    if desired_end >= len(joined):
        end = len(joined)
    else:
        # If desired_end falls on whitespace, cut there. Otherwise extend to the
        # next whitespace (within `max_extra`) so we don't split a word.
        if joined[desired_end].isspace():
            end = desired_end
        else:
            search_end = min(len(joined), desired_end + max_extra)
            m = re.search(r"\s", joined[desired_end:search_end])
            if m:
                end = desired_end + m.start()
            else:
                # No whitespace found within cap: fall back to desired_end
                end = desired_end

    # Safety: ensure end > start to avoid infinite loops
    if end <= start:
        end = min(len(joined), start + chunk_size)

    chunk_text = joined[start:end]

    # Map pages overlapping this chunk
    pages = set()
    for i, off in enumerate(offsets):
        page_start = off
        page_end = off + len(cleaned_pages[i])
        if not (page_end < start or page_start > end):
            pages.add(documents[i]['page'])

    chunks.append({
        'text': chunk_text,
        'start': start,
        'end': end,
        'pages': sorted(pages),
    })

    start += step

# Stats and samples
lengths = [len(c['text']) for c in chunks]
print(f"Created {len(chunks)} chunks (target {chunk_size}, overlap {chunk_overlap}, max_extra {max_extra}).")
print('Min chars:', min(lengths) if lengths else 0)
print('Max chars:', max(lengths) if lengths else 0)
print('Mean chars:', (sum(lengths)/len(lengths)) if lengths else 0)

for i, c in enumerate(chunks[:6]):
    print(f"\n--- Chunk {i+1} pages={c['pages']} chars={len(c['text'])} ---")
    print(c['text'][:300])

# Keep `chunks` in notebook state for downstream embedding/indexing


Created 303 chunks (target 800, overlap 200, max_extra 400).
Min chars: 270
Max chars: 850
Mean chars: 801.8580858085809

--- Chunk 1 pages=[10, 11, 12, 13, 14, 15, 16] chars=801 ---
What is Cloud Computing
Section
How websites work
Server
network
Clients have IP addresses
Servers have IP addresses
Client
Just like when you’re sending post mail!
What is a server composed of?
• Compute: CPU
• Memory: RAM
• Storage: Data
• Database: Store data in a structured way
• Network: Router

--- Chunk 2 pages=[14, 15, 16, 17] chars=802 ---
er / client on your network
Router
Switch
Traditionally, how to build infrastructure
Data center
Home or Garage
Office
Problems with traditional IT approach
• Pay for the rent for the data center
• Pay for power supply, cooling, maintenance
• Adding and replacing hardware takes time
• Scaling is lim

--- Chunk 3 pages=[17, 18, 19, 20] chars=806 ---
 a cloud services platform with pay-as-you-go pricing
• You can provision exactly the right type and size of comput

In [12]:
from pathlib import Path
import json

out_file = Path("Chunks/chunks.json")
out_file.parent.mkdir(parents=True, exist_ok=True)
with out_file.open("w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)

print(f"Saved {len(chunks)} chunks to {out_file}")


Saved 303 chunks to Chunks/chunks.json


In [ ]:
# Generate embeddings for chunks and save them
import numpy as np
from pathlib import Path

out_dir = Path("Embeddings-data")
out_dir.mkdir(parents=True, exist_ok=True)

texts = [c['text'] for c in chunks]

try:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer('all-MiniLM-L6-v2')
    print('Using sentence-transformers model:', model.__class__.__name__)
    embeddings = model.encode(texts, show_progress_bar=True, convert_to_numpy=True)
    np.save(out_dir / 'embeddings.npy', embeddings)
    print(f"Saved embeddings shape {embeddings.shape} to {out_dir / 'embeddings.npy'}")

    # Also attach embeddings lengths to chunks metadata (not storing vectors inline)
    for i, c in enumerate(chunks):
        c['embedding_index'] = i

    # Save chunks with embedding index for later lookup
    import json
    with open(out_dir / 'chunks_with_index.json', 'w', encoding='utf-8') as f:
        json.dump(chunks, f, ensure_ascii=False, indent=2)
    print(f"Saved chunks metadata to {out_dir / 'chunks_with_index.json'}")

except Exception as e:
    print('Could not run sentence-transformers embedding step:', e)


/home/abiram/Prayoga/Marga/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 390.83it/s]


Using sentence-transformers model: SentenceTransformer


Batches: 100%|██████████| 10/10 [00:07<00:00,  1.32it/s]

Saved embeddings shape (303, 384) to RAG Without Frameworks/embeddings.npy
Saved chunks metadata to RAG Without Frameworks/chunks_with_index.json


In [18]:
# Build a cosine-similarity FAISS index from saved embeddings
import json
from pathlib import Path

import numpy as np

try:
    import faiss
except Exception as e:
    print('FAISS is not available in this environment:', e)
    print('Install it with: uv add faiss-cpu')
else:
    out_dir = Path("Embeddings-data")
    embeddings_path = out_dir / "embeddings.npy"
    chunks_path = out_dir / "chunks_with_index.json"
    old_index_path = out_dir / "faiss.index"
    index_path = out_dir / "faiss_cosine.index"
    metadata_path = out_dir / "faiss_metadata.json"

    embeddings = np.load(embeddings_path).astype(np.float32)
    with chunks_path.open("r", encoding="utf-8") as f:
        chunks_for_index = json.load(f)

    # Normalize a copy so the original embeddings stay untouched.
    cosine_embeddings = embeddings.copy()
    faiss.normalize_L2(cosine_embeddings)

    dimension = cosine_embeddings.shape[1]
    index = faiss.IndexFlatIP(dimension)
    index.add(cosine_embeddings)

    # Replace any old L2-style index with the cosine version.
    if old_index_path.exists():
        old_index_path.unlink()
    faiss.write_index(index, str(index_path))

    with metadata_path.open("w", encoding="utf-8") as f:
        json.dump(chunks_for_index, f, ensure_ascii=False, indent=2)

    print(f"Built cosine FAISS index with {index.ntotal} vectors and dimension {dimension}")
    print(f"Saved index to {index_path}")
    print(f"Saved metadata to {metadata_path}")


Built cosine FAISS index with 303 vectors and dimension 384
Saved index to Embeddings-data/faiss_cosine.index
Saved metadata to Embeddings-data/faiss_metadata.json


# Query with FAISS

This section loads the cosine-similarity FAISS index, embeds a question, normalizes the query vector, and retrieves the top matching chunks.


In [ ]:
from pathlib import Path
import json

import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

out_dir = Path("Embeddings-data")
index_path = out_dir / "faiss_cosine.index"
metadata_path = out_dir / "faiss_metadata.json"

index = faiss.read_index(str(index_path))
with metadata_path.open("r", encoding="utf-8") as f:
    metadata = json.load(f)

model = SentenceTransformer("all-MiniLM-L6-v2")


def search(query: str, k: int = 5):
    query_embedding = model.encode([query], convert_to_numpy=True).astype(np.float32)
    faiss.normalize_L2(query_embedding)
    scores, indices = index.search(query_embedding, k)
    return scores[0], indices[0]


query = "What is Amazon S3?"
scores, indices = search(query, k=5)

print("Query:", query)
for rank, idx in enumerate(indices, start=1):
    if idx == -1:
        continue
    item = metadata[idx]
    print("=" * 60)
    print("Rank:", rank)
    print("Similarity:", float(scores[rank - 1]))
    print("Pages:", item.get("pages"))
    print(item["text"][:500])


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 432.83it/s]


Query: What is Amazon S3?
Rank: 1
Similarity: 0.6821873784065247
Pages: [146, 147, 148]
11 9’s) of objects across multiple AZ
• If you store 10,000,000 objects with Amazon S3, you can on average expect to
incur a loss of a single object once every 10,000 years
• Same for all storage classes
• Availability:
• Measures how readily available a service is
• Varies depending on storage class
• Example: S3 standard has 99.99% availability = not available 53 minutes a year
S3 Standard – General Purpose
• 99.99% Availability
• Used for frequently accessed data
• Low latency and high throug
Rank: 2
Similarity: 0.6737831830978394
Pages: [130, 131, 132]
ne of the main building blocks of AWS
• It’s advertised as ”infinitely scaling” storage
• Many websites use Amazon S3 as a backbone
• Many AWS services use Amazon S3 as an integration as well
• We’ll have a step-by-step approach to S3
Amazon S3 Use cases
• Backup and storage
• Disaster Recovery
• Archive
• Hybrid Cloud storage
• Application hostin

## Testing Sematic Retrival Capability

In [22]:
query = "What AWS service can I use to store backups?"
scores, indices = search(query, k=5)

print("Query:", query)
for rank, idx in enumerate(indices, start=1):
    if idx == -1:
        continue
    item = metadata[idx]
    print("=" * 60)
    print("Rank:", rank)
    print("Similarity:", float(scores[rank - 1]))
    print("Pages:", item.get("pages"))
    print(item["text"][:500])


Query: What AWS service can I use to store backups?
Rank: 1
Similarity: 0.6923208236694336
Pages: [443, 444, 445, 446]
Backup
AWS Backup
Create Backup Plan
(frequency, retention
policy)
Assign AWS Resources
Amazon EC2
Amazon EBS
DynamoDB
Amazon RDS
Amazon EFS
Amazon Aurora
Amazon FSx
AWS Storage
Gateway
Amazon S3
Automatically
backed up to
Disaster Recovery Strategies
Backup and Restore
AWS Cloud
Corporate Data Center
Servers
S3
Pilot Light
AWS Cloud
Corporate Data Center
Web Server
EC2
Core functions of the app
Ready to scale, but minimal setup
Cost
Cost
Disaster Recovery Strategies
Multi-Site / Hot-Site
AWS Cl
Rank: 2
Similarity: 0.6344842910766602
Pages: [147, 148, 149]
cations, content
distribution…
S3 Storage Classes – Infrequent Access
• For data that is less frequently accessed, but requires rapid access when needed
• Lower cost than S3 Standard
• Amazon S3 Standard-Infrequent Access (S3 Standard-IA)
• 99.9% Availability
• Use cases: Disaster Recovery, backups
• Amazon S3 One Zo

# Generate an Answer with Ollama

This cell turns the retrieved chunks into a context block and sends a prompt to `qwen3:8b` through Ollama.


In [26]:
from ollama import chat

query = "How to choose an AWS Region?"

context = "\n\n".join(
    metadata[idx]["text"]
    for idx in indices
    if idx != -1
)

prompt = f"""
Answer the question using only the provided context.

Context:
{context}

Question:
{query}

Answer:
"""

response = chat(
    model="qwen3:8b",
    messages=[
        {
            "role": "user",
            "content": prompt,
        }
    ],
)

print(response.message.content)


To choose an AWS Region, consider the following based on the context:  
1. **Proximity to Users**: Select a region close to your target audience to minimize latency. For example, **N. Virginia (us-east-1)** is a commonly used region.  
2. **Compliance and Data Residency**: Ensure the region complies with legal requirements for data storage (e.g., GDPR, HIPAA).  
3. **Availability Zones (AZs)**: Use multiple AZs within a region for redundancy (e.g., **Availability Zone - A** and **Availability Zone - B** in the context).  
4. **Disaster Recovery**: Implement cross-region backups (e.g., **Cross-Region Backup**) to ensure data resilience.  
5. **Cost**: Evaluate pricing differences across regions for services like S3 storage classes (e.g., **S3 Standard-IA** or **S3 Glacier**).  
6. **Service Availability**: Verify that required services (e.g., EC2, S3) are supported in the chosen region.  

The context highlights **N. Virginia (us-east-1)** as an example region and emphasizes the importa